## 9. Summary Table: Flow Phases & Key Components

| Phase | Frontend Component | Backend Endpoint | Database Tables | WebSocket | Key Files |
|-------|-------------------|------------------|-----------------|-----------|-----------|
| **① User Creation** | RegisterHead.jsx | POST /admin/heads | users, head_profile | N/A | adminService.js, admin.py |
| **② Authentication** | Login.jsx | POST /auth/login | users | N/A | authService.js, auth.py |
| **③ Session Creation** | Head Dashboard | POST /sessions | sessions, session_participants | N/A | headService.js, sessions.py |
| **④ WebSocket Connect** | TrainerSession.jsx, LearnerSession.jsx | WS /ws/rtc/{id} | session_participants | ✓ JOIN | videoConferenceService.js, websocket.py |
| **⑤ WebRTC Offer/Answer** | Both components | N/A (via WebSocket) | N/A | ✓ OFFER, ANSWER | videoConferenceService.js, websocket.py |
| **⑥ Media Streaming** | Video elements | N/A (P2P) | session_logs | ✓ ICE Candidates | videoConferenceService.js (ontrack) |
| **⑦ Session Active** | UI state updates | PUT /sessions | session_notes | ✓ PING/PONG | TrainerSession.jsx, sessions.py |
| **⑧ Termination** | Disconnect UI | N/A (via WebSocket) | sessions | ✓ CLOSE | videoConferenceService.js, websocket.py |

---

## 10. API Request/Response Examples

### Example 1: User Creation
```json
REQUEST: POST /admin/heads
{
  "name": "Dr. John Doe",
  "email": "john@example.com",
  "age": 35,
  "contact_no": "+1234567890",
  "address": "123 Main St"
}

RESPONSE: 201 Created
{
  "head_id": 42,
  "user_id": 123,
  "passcode": "456789",
  "email": "john@example.com",
  "name": "Dr. John Doe"
}
```

### Example 2: Login
```json
REQUEST: POST /auth/user/login
{
  "role": "head",
  "passcode": "456789"
}

RESPONSE: 200 OK
{
  "access_token": "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9...",
  "token_type": "bearer",
  "role": "head",
  "user_id": 123,
  "name": "Dr. John Doe"
}
```

### Example 3: Session Creation
```json
REQUEST: POST /sessions/create
{
  "trainer_id": 5,
  "learner_id": 23,
  "scheduled_datetime": "2024-02-15T14:00:00",
  "topic": "DLB Assessment",
  "notes": "Follow-up session"
}

RESPONSE: 201 Created
{
  "session_id": "550e8400-e29b-41d4-a716-446655440000",
  "status": "scheduled",
  "trainer_name": "John Trainer",
  "learner_name": "Jane Learner",
  "scheduled_datetime": "2024-02-15T14:00:00"
}
```

### Example 4: WebSocket Join Message
```json
CLIENT SEND:
{
  "type": "join",
  "role": "trainer",
  "sessionId": "550e8400-e29b-41d4-a716-446655440000"
}

SERVER SEND:
{
  "type": "connected",
  "role": "trainer",
  "session_id": "550e8400-e29b-41d4-a716-446655440000",
  "heartbeat_interval": 30
}

SERVER SEND (when both joined):
{
  "type": "peer_ready",
  "role": "both_connected"
}
```

### Example 5: WebRTC Offer
```json
CLIENT SEND:
{
  "type": "offer",
  "offer": {
    "type": "offer",
    "sdp": "v=0\r\no=... (SDP content)"
  }
}
```

---

## 11. Performance & Timing Metrics

```
OPERATION                          EXPECTED TIME    CRITICAL?
─────────────────────────────────────────────────────────────
User Registration                  < 2s             No
Login Authentication               < 1s             Yes
Session Creation                   < 3s             No
WebSocket Connection               < 2s             Yes
Heartbeat Ping/Pong               30s interval       Yes (keeps alive)
WebRTC Offer Creation             < 1s             Yes
Answer Creation & Send            < 2s             Yes
ICE Candidate Exchange            < 10s             Yes
Total P2P Connection Time         < 15s             Yes
Media Streaming Start             < 2s after P2P   Yes

Database Query (by index)          < 100ms          Yes
Database Query (without index)     < 1s (slowlog)   No (needs optimization)
File Upload (1MB)                 < 5s             No
```

---

## 12. Key Security Measures

```
LAYER                              SECURITY MEASURE
─────────────────────────────────────────────────────
Frontend
├─ Login Form                      - Rate limit display (disable after 5 attempts)
├─ Token Storage                   - Stored in localStorage (no XSS protection shown)
└─ API Calls                       - Authorization header with Bearer token

Backend
├─ Authentication                  - Passcode hash verification + JWT
├─ Authorization                   - Role-based access control (admin/head/trainer/learner)
├─ Rate Limiting                   - 5 login attempts per minute per IP
├─ Input Validation                - Email format, phone format, age range
├─ SQL Injection Prevention        - SQLAlchemy ORM (parameterized queries)
├─ HTTPS/WSS                       - Encrypted transport layer
├─ CORS                            - Cross-origin requests restricted
└─ Logging                         - All auth attempts logged with IP & timestamp

WebSocket
├─ Message Relay                   - Only relay to authenticated peers
├─ Heartbeat                       - Detect dead connections (90s timeout)
├─ Connection Tracking             - Store conn_info with metadata
└─ Role Validation                 - Only trainer can initiate offer

WebRTC
├─ STUN/TURN                       - Public servers with no credentials
├─ Encryption                      - DTLS-SRTP (media encryption by default)
├─ ICE Candidates                  - Filter private IP addresses
└─ P2P Only                        - Data never passes through server

Database
├─ Passcode Storage                - Hashed + Encrypted (double protection)
├─ Role Enforcement                - Foreign key constraints
├─ Audit Trails                    - created_at, updated_at timestamps
└─ Soft Deletes                    - is_active flag instead of hard delete
```

---

## 13. Scalability Considerations

```
CURRENT BOTTLENECKS & IMPROVEMENTS

1. WebSocket Message Relay
   Current: Single backend server handles all connections
   Improvement: Use Redis Pub/Sub for multi-server deployment
   
2. Session Lookup
   Current: Dictionary in memory (rtc_connections)
   Improvement: Store in Redis for cross-server access
   
3. Database Queries
   Current: Some queries without indexes (slow logs detected)
   Improvement: Add composite indexes on (session_id, role)
   
4. Media Streaming
   Current: P2P works well, TURN server needed for NAT traversal
   Improvement: Scale TURN server instances geographically
   
5. File Uploads
   Current: Local filesystem storage
   Improvement: Use S3 for distributed file storage
   
6. Horizontal Scaling
   Current: Not applicable (WebSocket connection tied to single server)
   Improvement: Implement sticky sessions or Redis connection manager
```

---

## 8. Error Handling & Recovery Flow

```
┌──────────────────────────────────────────────────────────────────────┐
│                ERROR HANDLING & RECOVERY FLOW                         │
├──────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  AUTHENTICATION ERRORS                                                │
│  ├─ 401 Unauthorized (Invalid passcode)                              │
│  │   ├─ Frontend: Show toast "Incorrect role or passcode"            │
│  │   ├─ Clear form fields                                            │
│  │   ├─ Backend logs: Log failed attempt with IP                     │
│  │   └─ Rate limit: Increment attempt counter                        │
│  │                                                                    │
│  ├─ 429 Too Many Requests (Rate limited)                             │
│  │   ├─ Frontend: Show toast "Too many login attempts"               │
│  │   ├─ Disable login button for 60 seconds                          │
│  │   └─ Backend: Block requests from IP for 1 minute                 │
│  │                                                                    │
│  └─ 500 Server Error                                                 │
│      ├─ Frontend: Show toast "Server error, try again"               │
│      ├─ Retry logic: Exponential backoff                             │
│      └─ Backend: Log error with traceback                            │
│                                                                        │
│  SESSION CREATION ERRORS                                              │
│  ├─ Trainer/Learner not found                                        │
│  │   ├─ Frontend: Show toast "Selected user not found"               │
│  │   └─ Reload users list from API                                   │
│  │                                                                    │
│  ├─ User not active                                                  │
│  │   ├─ Frontend: Show toast "User account is inactive"              │
│  │   └─ Suggest contacting admin                                     │
│  │                                                                    │
│  └─ Duplicate session at same time                                   │
│      ├─ Frontend: Show toast "Trainer/Learner busy at that time"    │
│      └─ Suggest alternate time slots                                 │
│                                                                        │
│  WEBSOCKET CONNECTION ERRORS                                          │
│  ├─ Connection failed                                                │
│  │   ├─ Frontend: Auto-retry with exponential backoff                │
│  │   │   - Attempt 1: 1 second                                       │
│  │   │   - Attempt 2: 2 seconds                                      │
│  │   │   - Attempt 3: 4 seconds                                      │
│  │   │   - Attempt 4: 8 seconds                                      │
│  │   │   - Max retry delay: 30 seconds                               │
│  │   │                                                                │
│  │   ├─ UI: Show "Connecting..." spinner                             │
│  │   ├─ Max attempts: Infinity (keep retrying)                       │
│  │   └─ User can manually close modal to stop retrying               │
│  │                                                                    │
│  ├─ Heartbeat timeout (no pong for 90s)                             │
│  │   ├─ Backend: Set conn_info.is_alive = False                     │
│  │   ├─ Backend: Cancel heartbeat_task                              │
│  │   ├─ Backend: Notify other peer of disconnection                │
│  │   ├─ Frontend: Receive "peer_disconnected" message               │
│  │   └─ Frontend: Show "Connection lost" message                    │
│  │                                                                    │
│  └─ Invalid message format                                           │
│      ├─ Backend: Log warning with message details                   │
│      └─ Continue processing (ignore malformed messages)              │
│                                                                        │
│  WEBRTC CONNECTION ERRORS                                            │
│  ├─ getUserMedia denied                                              │
│  │   ├─ Error: NotAllowedError (User denied permission)             │
│  │   ├─ Frontend: Show toast "Camera/mic access denied"              │
│  │   ├─ Instructions: Check browser permissions                      │
│  │   ├─ Fallback: Try audio-only mode                                │
│  │   └─ If still fails, show cannot join session                    │
│  │                                                                    │
│  ├─ getUserMedia not available                                       │
│  │   ├─ Error: NotFoundError (No devices found)                      │
│  │   ├─ Frontend: Show toast "No camera/mic found"                   │
│  │   └─ Fallback: Try audio-only mode                                │
│  │                                                                    │
│  ├─ Overconstrained error                                            │
│  │   ├─ Error: OverconstrainedError (Device doesn't meet specs)      │
│  │   ├─ Resolution: Lower constraints (e.g., 480p instead of 720p)   │
│  │   ├─ Retry with relaxed constraints                               │
│  │   └─ Fallback: Audio-only mode                                    │
│  │                                                                    │
│  ├─ Offer/Answer creation failed                                     │
│  │   ├─ Error: Possible ICE gathering timeout                        │
│  │   ├─ Frontend: Retry offer creation after 2 seconds               │
│  │   ├─ Max retry attempts: 3                                        │
│  │   └─ If fails: Show "Connection failed, try again"                │
│  │                                                                    │
│  ├─ ICE candidate addition failed                                    │
│  │   ├─ Error: "Failed to execute 'addIceCandidate'"                 │
│  │   ├─ Possible cause: Remote description not set                   │
│  │   ├─ Solution: Queue candidate and retry later                    │
│  │   └─ Continue processing other candidates                         │
│  │                                                                    │
│  ├─ Connection stuck in 'checking' state                             │
│  │   ├─ Timeout: No candidates added for 30 seconds                  │
│  │   ├─ Possible cause: Firewall blocking UDP                        │
│  │   ├─ Solution: TURN server should provide relay                   │
│  │   ├─ If no candidates: Cannot establish P2P                       │
│  │   └─ Show "Connection failed, check network"                      │
│  │                                                                    │
│  └─ Connection failed after initial success                          │
│      ├─ Detect: onconnectionstatechange event = 'failed'             │
│      ├─ Log: Detailed failure reason from iceConnectionState         │
│      ├─ Notify: Show toast "Connection lost"                         │
│      └─ Recovery: Auto-retry WebSocket connection                    │
│                                                                        │
│  DATABASE ERRORS                                                      │
│  ├─ Duplicate email                                                  │
│  │   ├─ Error: Unique constraint violation                           │
│  │   ├─ Frontend: Show toast "Email already registered"              │
│  │   └─ Suggest: Use different email or recover account              │
│  │                                                                    │
│  ├─ Invalid foreign key                                              │
│  │   ├─ Error: Trainer/Learner doesn't exist                         │
│  │   ├─ Backend: Validate IDs before insertion                       │
│  │   └─ Frontend: Refresh users list                                 │
│  │                                                                    │
│  └─ Timeout                                                          │
│      ├─ Query took > 30 seconds                                      │
│      ├─ Backend: Log slow query                                      │
│      ├─ Frontend: Show "Server is slow, please retry"                │
│      └─ Consider: Index optimization needed                          │
│                                                                        │
│  RECOVERY STRATEGIES                                                 │
│  ├─ Exponential Backoff: Increase delay between retries              │
│  ├─ Circuit Breaker: Stop retrying after threshold                   │
│  ├─ Health Checks: Periodic server status checks                     │
│  ├─ Graceful Degradation: Continue without non-critical features     │
│  ├─ User Notification: Clear error messages with actions              │
│  ├─ Logging: All errors logged with context for debugging            │
│  ├─ Monitoring: Alerts for high error rates                          │
│  └─ Fallbacks: Audio-only, no video, text-only chat                  │
│                                                                        │
└──────────────────────────────────────────────────────────────────────┘
```

---

## 7. Component Interaction Diagram

```
┌─────────────────────────────────────────────────────────────────────────┐
│          FRONTEND/BACKEND/DATABASE COMPONENT INTERACTIONS                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                           │
│  FRONTEND (React Components)                                              │
│  ┌───────────────────────────────────────────────────────────────────┐  │
│  │ ┌─────────────────┐  ┌──────────────────┐  ┌────────────────────┐ │  │
│  │ │ Login.jsx       │  │ RegisterHead.jsx │  │ TrainerSession.jsx │ │  │
│  │ │ - Role select   │  │ - Form fields    │  │ - Video elements  │ │  │
│  │ │ - Passcode input│  │ - Validation     │  │ - Whiteboard      │ │  │
│  │ └────────┬────────┘  └─────────┬────────┘  └────────┬──────────┘ │  │
│  │          │                     │                    │             │  │
│  │          └─────────────────────┼────────────────────┘             │  │
│  │                                │                                   │  │
│  │         AuthContext (Global State)                                │  │
│  │         - user: { id, role, name }                              │  │
│  │         - isAuthenticated: boolean                              │  │
│  │         - loading: boolean                                      │  │
│  │                                │                                   │  │
│  └────────────────────────────────┼───────────────────────────────────┘  │
│                                   │                                       │
│  API SERVICES (Axios Clients)      │                                       │
│  ┌────────────────────────────────┼───────────────────────────────────┐  │
│  │ ┌──────────────┐  ┌──────────┐ │ ┌─────────────────────────────┐ │  │
│  │ │authService   │  │headService│ │ │ videoConferenceService.js │ │  │
│  │ │- userLogin() │  │- register │  │ │ - connectWebSocket()      │ │  │
│  │ │- logout()    │  │  Head()   │  │ │ - createPeerConnection()  │ │  │
│  │ └──────────────┘  └──────────┘  │ │ - createOffer/Answer      │ │  │
│  │                                  │ │ - handleSignaling()       │ │  │
│  │  Requests Headers:               │ └─────────────────────────────┘ │  │
│  │  - Authorization: Bearer <token> │                                  │  │
│  │  - Content-Type: application/json│                                  │  │
│  │                                  │                                  │  │
│  └──────────────────────────────────┼──────────────────────────────────┘  │
│                                     │                                      │
│                                     ▼                                      │
│                        BACKEND APIs (FastAPI)                             │
│  ┌───────────────────────────────────────────────────────────────────┐   │
│  │ ┌──────────────┐  ┌──────────┐  ┌──────────┐  ┌─────────────┐   │   │
│  │ │ /auth/login  │  │/admin/   │  │/sessions/│  │/ws/rtc/{id} │   │   │
│  │ │ - Passcode   │  │heads     │  │create    │  │- WebSocket  │   │   │
│  │ │   verify     │  │- Create  │  │- Validate│  │  Signaling  │   │   │
│  │ │ - JWT token  │  │  Head    │  │  params  │  │- Message    │   │   │
│  │ │ - Return user│  │- Generate│  │- Generate│  │  relay      │   │   │
│  │ │   data       │  │  passcode│  │  session │  │- Heartbeat  │   │   │
│  │ └──────────────┘  └──────────┘  └──────────┘  └─────────────┘   │   │
│  │                                                                    │   │
│  │  Dependency Injection (FastAPI):                                 │   │
│  │  - get_db: AsyncSession (Database connection)                    │   │
│  │  - get_current_user: User (JWT validation)                       │   │
│  │  - get_current_head: HeadProfile (Role check)                    │   │
│  │                                                                    │   │
│  └────────────────┬───────────────────────────────┬──────────────────┘   │
│                   │                               │                      │
│                   ▼                               ▼                      │
│           CRUD Operations             WebSocket Connection Manager       │
│  ┌────────────────────────────────┐  ┌──────────────────────────────┐   │
│  │ user_crud.py                   │  │ rtc_connections = {          │   │
│  │ - create_head()                │  │   session_id: {              │   │
│  │ - create_learner()             │  │     'trainer': ConnectionInfo│   │
│  │ - create_trainer()             │  │     'learner': ConnectionInfo│   │
│  │ - get_user_by_id()             │  │   }                          │   │
│  │ - update_user()                │  │ }                            │   │
│  │ - verify_passcode()            │  │                              │   │
│  │                                │  │ async send_heartbeat(conn)   │   │
│  │ head_crud.py                   │  │ async cleanup_connection()   │   │
│  │ - create_session()             │  │ async handle_signaling()     │   │
│  │ - get_active_sessions()        │  │                              │   │
│  │ - update_session_status()      │  │ Message Relay:               │   │
│  │                                │  │ trainer -> learner           │   │
│  │ sessions_crud.py               │  │ learner -> trainer           │   │
│  │ - create()                     │  │ trainer -> observer          │   │
│  │ - update_status()              │  │                              │   │
│  │ - get_by_id()                  │  └──────────────────────────────┘   │
│  └────────────────┬────────────────┘                                     │
│                   │                                                      │
│                   ▼                                                      │
│          DATABASE (PostgreSQL/SQLAlchemy)                               │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ ┌──────────┐  ┌─────────┐  ┌──────────┐  ┌─────────────────┐   │  │
│  │ │ users    │  │head_    │  │sessions  │  │session_         │   │  │
│  │ │- id      │  │profile  │  │- id      │  │participants     │   │  │
│  │ │- name    │  │- age    │  │- trainer │  │- session_id     │   │  │
│  │ │- email   │  │- address│  │- learner │  │- user_id        │   │  │
│  │ │- role    │  │- phone  │  │- status  │  │- role           │   │  │
│  │ │- passcode│  │- exp    │  │- created │  │- joined_at      │   │  │
│  │ └──────────┘  └─────────┘  └──────────┘  └─────────────────┘   │  │
│  │                                                                  │  │
│  │ ┌────────────┐  ┌─────────┐  ┌──────────────┐                  │  │
│  │ │trainer_    │  │learner_ │  │session_notes │                  │  │
│  │ │profile     │  │profile  │  │- session_id  │                  │  │
│  │ │- spec      │  │- dob    │  │- notes_text  │                  │  │
│  │ │- exp       │  │- age    │  │- created_by  │                  │  │
│  │ │- avail     │  │- assess │  │- created_at  │                  │  │
│  │ └────────────┘  └─────────┘  └──────────────┘                  │  │
│  │                                                                  │  │
│  └──────────────────────────────────────────────────────────────────┘  │
│                                                                           │
└─────────────────────────────────────────────────────────────────────────┘
```

**Data Flow Example: User Creation**

```
Admin Form Input
       ↓
RegisterHead.jsx validation
       ↓
adminService.registerHead(formData)
       ↓
POST /admin/heads with Authorization header
       ↓
Backend auth.py: get_current_user (JWT validation)
       ↓
user_crud.py: create_head(db, data, admin_id)
       ↓
SQLAlchemy ORM creates User + HeadProfile records
       ↓
Database INSERT statements executed
       ↓
Return response to frontend
       ↓
localStorage.setItem('user_data')
       ↓
Frontend redirect to /admin/manage-heads
```

---

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
import numpy as np

# Create comprehensive application flow diagram
fig, ax = plt.subplots(figsize=(20, 14))
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.axis('off')

# Define colors
color_frontend = '#E3F2FD'  # Light blue
color_backend = '#FFF3E0'   # Light orange
color_database = '#F3E5F5'  # Light purple
color_websocket = '#E8F5E9' # Light green
color_webrtc = '#FCE4EC'    # Light pink

def draw_box(ax, x, y, width, height, text, color, fontsize=9):
    """Draw a rounded rectangle box"""
    box = FancyBboxPatch((x, y), width, height, boxstyle="round,pad=0.1",
                          edgecolor='black', facecolor=color, linewidth=2)
    ax.add_patch(box)
    ax.text(x + width/2, y + height/2, text, ha='center', va='center',
            fontsize=fontsize, weight='bold', wrap=True)

def draw_arrow(ax, x1, y1, x2, y2, label='', color='black'):
    """Draw an arrow between two points"""
    arrow = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='->', 
                           mutation_scale=20, color=color, linewidth=2)
    ax.add_patch(arrow)
    if label:
        mid_x, mid_y = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mid_x + 2, mid_y, label, fontsize=8, style='italic', color=color)

# Title
ax.text(50, 98, 'HurshBinTech PWA - Complete Application Flow Diagram', 
        ha='center', fontsize=16, weight='bold')

# Phase 1: User Creation (Left side)
ax.text(15, 92, '① USER CREATION', ha='center', fontsize=11, weight='bold', 
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', linewidth=2))
draw_box(ax, 2, 85, 8, 4, 'Admin Portal\nRegister Head', color_frontend)
draw_arrow(ax, 6, 85, 6, 80)
draw_box(ax, 2, 76, 8, 4, 'Form Validation\n(Frontend)', color_frontend)
draw_arrow(ax, 6, 76, 6, 71)
draw_box(ax, 2, 67, 8, 4, 'POST /admin/heads\n(Backend)', color_backend)
draw_arrow(ax, 6, 67, 6, 62)
draw_box(ax, 2, 58, 8, 4, 'Generate Passcode\nCreate Records', color_backend)
draw_arrow(ax, 6, 58, 6, 53)
draw_box(ax, 2, 49, 8, 4, 'INSERT User\nINSERT HeadProfile', color_database)

# Phase 2: Authentication (Left side continued)
ax.text(15, 45, '② AUTHENTICATION', ha='center', fontsize=11, weight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', linewidth=2))
draw_arrow(ax, 6, 49, 6, 42)
draw_box(ax, 2, 38, 8, 4, 'Login Page\nSelect Role', color_frontend)
draw_arrow(ax, 6, 38, 6, 33)
draw_box(ax, 2, 29, 8, 4, 'POST /auth/login\nRate Limited', color_backend)
draw_arrow(ax, 6, 29, 6, 24)
draw_box(ax, 2, 20, 8, 4, 'Verify Passcode\nCreate JWT', color_backend)
draw_arrow(ax, 6, 20, 6, 15)
draw_box(ax, 2, 11, 8, 4, 'localStorage Token\nRedirect Dashboard', color_frontend)

# Phase 3: Session Creation (Middle-left)
ax.text(35, 92, '③ SESSION CREATION', ha='center', fontsize=11, weight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', linewidth=2))
draw_box(ax, 22, 85, 8, 4, 'Head Dashboard\nCreate Session', color_frontend)
draw_arrow(ax, 26, 85, 26, 80)
draw_box(ax, 22, 76, 8, 4, 'Select Trainer\n& Learner', color_frontend)
draw_arrow(ax, 26, 76, 26, 71)
draw_box(ax, 22, 67, 8, 4, 'POST /sessions\nCreate', color_backend)
draw_arrow(ax, 26, 67, 26, 62)
draw_box(ax, 22, 58, 8, 4, 'Generate UUID\nValidate Users', color_backend)
draw_arrow(ax, 26, 58, 26, 53)
draw_box(ax, 22, 49, 8, 4, 'INSERT Session\nSET status=scheduled', color_database)
draw_arrow(ax, 26, 49, 26, 44)
draw_box(ax, 22, 40, 8, 4, 'Send Notifications\n(Email)', color_backend)

# Phase 4: WebSocket Connection (Middle)
ax.text(55, 92, '④ WEBSOCKET CONNECTION', ha='center', fontsize=11, weight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', linewidth=2))
draw_box(ax, 42, 85, 8, 4, 'Trainer/Learner\nJoin Session', color_frontend)
draw_arrow(ax, 46, 85, 46, 80)
draw_box(ax, 42, 76, 8, 4, 'Connect WebSocket\nwss://...', color_websocket)
draw_arrow(ax, 46, 76, 46, 71)
draw_box(ax, 42, 67, 8, 4, 'Send JOIN Message\n{role, sessionId}', color_websocket)
draw_arrow(ax, 46, 67, 46, 62)
draw_box(ax, 42, 58, 8, 4, 'Backend: Accept\nStart Heartbeat', color_backend)
draw_arrow(ax, 46, 58, 46, 53)
draw_box(ax, 42, 49, 8, 4, 'Store Connection\nin rtc_connections', color_backend)
draw_arrow(ax, 46, 49, 46, 44)
draw_box(ax, 42, 40, 8, 4, 'Send "peer_ready"\nwhen both join', color_websocket)

# Phase 5: WebRTC Setup (Middle-right)
ax.text(75, 92, '⑤ WEBRTC SETUP', ha='center', fontsize=11, weight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', linewidth=2))
draw_box(ax, 62, 85, 8, 4, 'Create Peer\nConnections', color_webrtc)
draw_arrow(ax, 66, 85, 66, 80)
draw_box(ax, 62, 76, 8, 4, 'Add Local Tracks\ngetUserMedia()', color_webrtc)
draw_arrow(ax, 66, 76, 66, 71)
draw_box(ax, 62, 67, 8, 4, 'Create Offer\n(Trainer)', color_webrtc)
draw_arrow(ax, 66, 67, 66, 62)
draw_box(ax, 62, 58, 8, 4, 'Send via WebSocket\n(Signaling)', color_websocket)
draw_arrow(ax, 66, 58, 66, 53)
draw_box(ax, 62, 49, 8, 4, 'Create Answer\n(Learner)', color_webrtc)
draw_arrow(ax, 66, 49, 66, 44)
draw_box(ax, 62, 40, 8, 4, 'Exchange ICE\nCandidates', color_webrtc)

# Phase 6: Media Streaming (Right side)
ax.text(85, 85, '⑥ MEDIA\nSTREAMING', ha='center', fontsize=11, weight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', linewidth=2))
draw_box(ax, 80, 76, 8, 4, 'P2P Connection\nEstablished', color_webrtc)
draw_arrow(ax, 84, 76, 84, 71)
draw_box(ax, 80, 67, 8, 4, 'Local Audio/Video\nFlowing', color_webrtc)
draw_arrow(ax, 84, 67, 84, 62)
draw_box(ax, 80, 58, 8, 4, 'Remote Tracks\nReceived', color_webrtc)
draw_arrow(ax, 84, 58, 84, 53)
draw_box(ax, 80, 49, 8, 4, 'Video Elements\nUpdated', color_frontend)
draw_arrow(ax, 84, 49, 84, 44)
draw_box(ax, 80, 40, 8, 4, 'Heartbeat Ping/Pong\nEvery 30s', color_websocket)

# Phase 7: Session Data Logging (Bottom center-left)
ax.text(35, 35, '⑦ DATA LOGGING', ha='center', fontsize=11, weight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', linewidth=2))
draw_box(ax, 22, 28, 8, 4, 'Update Session Status\nto "in-progress"', color_database)
draw_arrow(ax, 26, 28, 26, 23)
draw_box(ax, 22, 19, 8, 4, 'Log Session Metrics\nConnection Quality', color_database)

# Phase 8: Termination (Bottom center-right)
ax.text(55, 35, '⑧ TERMINATION', ha='center', fontsize=11, weight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', linewidth=2))
draw_box(ax, 42, 28, 8, 4, 'User Clicks\nEnd Session', color_frontend)
draw_arrow(ax, 46, 28, 46, 23)
draw_box(ax, 42, 19, 8, 4, 'Close WebSocket\nClean Resources', color_websocket)
draw_arrow(ax, 46, 19, 46, 14)
draw_box(ax, 42, 10, 8, 4, 'UPDATE Session\nstatus=completed', color_database)

# Add connection arrows between phases
draw_arrow(ax, 10, 80, 22, 80, color='gray')  # Creation to Auth
draw_arrow(ax, 30, 80, 42, 80, color='gray')  # Auth to Session
draw_arrow(ax, 50, 80, 62, 80, color='gray')  # Session to WebSocket
draw_arrow(ax, 70, 80, 80, 80, color='gray')  # WebSocket to WebRTC

# Add legend
legend_y = 6
ax.text(2, legend_y + 2, 'COLOR LEGEND:', fontsize=10, weight='bold')
draw_box(ax, 2, legend_y - 1, 4, 1, 'Frontend', color_frontend, fontsize=8)
draw_box(ax, 7, legend_y - 1, 4, 1, 'Backend', color_backend, fontsize=8)
draw_box(ax, 12, legend_y - 1, 4, 1, 'Database', color_database, fontsize=8)
draw_box(ax, 17, legend_y - 1, 4, 1, 'WebSocket', color_websocket, fontsize=8)
draw_box(ax, 22, legend_y - 1, 4, 1, 'WebRTC', color_webrtc, fontsize=8)

plt.tight_layout()
plt.savefig('application_flow_diagram.png', dpi=150, bbox_inches='tight')
print("✓ Application flow diagram created and saved!")
plt.show()

## 6. Session Termination & Cleanup Flow

```
┌────────────────────────────────────────────────────────────────────────┐
│           SESSION TERMINATION & CLEANUP FLOW                            │
├────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  DISCONNECT TRIGGER                                                      │
│  ├─ 1. User clicks "End Session" button                                │
│  ├─ 2. User closes browser/tab                                         │
│  ├─ 3. Network connection drops                                        │
│  ├─ 4. Heartbeat timeout (no pong for 90s)                            │
│  └─ 5. Server crash/restart                                           │
│                                                                          │
│     ▼                                                                    │
│  FRONTEND CLEANUP (VideoConferenceService.cleanup())                    │
│  ├─ localStream.getTracks().forEach(t => t.stop())                    │
│  │   └─ Release camera/microphone hardware access                      │
│  │                                                                      │
│  ├─ remoteStream.getTracks().forEach(t => t.stop())                   │
│  │   └─ Stop processing remote tracks                                  │
│  │                                                                      │
│  ├─ peerConnection.close()                                            │
│  │   └─ Close all senders/receivers                                    │
│  │                                                                      │
│  ├─ ws.close()                                                         │
│  │   └─ Close WebSocket connection                                     │
│  │                                                                      │
│  └─ Cancel all timers/intervals                                        │
│     └─ connectionMonitoringTimer, heartbeatTimer, etc.                │
│                                                                          │
│     ▼                                                                    │
│  WEBSOCKET CLOSE EVENT                                                  │
│  ├─ Browser sends WebSocket CLOSE frame                                │
│  └─ Message routed to backend websocket_rtc_endpoint()                │
│                                                                          │
│     ▼                                                                    │
│  BACKEND CLEANUP (websocket.py finally block)                          │
│  ├─ await cleanup_connection(rtc_connections, session_id, role)       │
│  │   │                                                                  │
│  │   ├─ Retrieve conn_info from rtc_connections[session_id][role]    │
│  │   │                                                                  │
│  │   ├─ Set conn_info.is_alive = False                               │
│  │   │   └─ Stops heartbeat_task loop                                 │
│  │   │                                                                  │
│  │   ├─ Cancel heartbeat_task                                         │
│  │   │   └─ task.cancel() → asyncio.CancelledError caught           │
│  │   │                                                                  │
│  │   └─ Delete from rtc_connections[session_id][role]                │
│  │       └─ If empty dict, delete rtc_connections[session_id]        │
│  │                                                                      │
│  │                                                                      │
│  ├─ NOTIFY OTHER PARTICIPANTS                                          │
│  │   │                                                                  │
│  │   ├─ If trainer left:                                              │
│  │   │   ├─ Notify learner: {"type": "peer_disconnected"}            │
│  │   │   └─ Close learner connection                                  │
│  │   │                                                                  │
│  │   ├─ If learner left:                                              │
│  │   │   ├─ Notify trainer: {"type": "peer_disconnected"}            │
│  │   │   └─ Close trainer connection                                  │
│  │   │                                                                  │
│  │   ├─ If observer left:                                             │
│  │   │   └─ Notify trainer: {"type": "observer_left"}               │
│  │   │                                                                  │
│  │   └─ If trainer left and observer present:                         │
│  │       └─ Notify observer: {"type": "trainer_disconnected"}        │
│  │                                                                      │
│  └─ LOG SESSION COMPLETION                                             │
│     ├─ logger.info(f"Cleaned up connection: {role}")                  │
│     └─ logger.info(f"Remaining connections in session: ...")          │
│                                                                          │
│     ▼                                                                    │
│  DATABASE UPDATE                                                         │
│  ├─ SELECT * FROM sessions WHERE id = session_id                      │
│  │   │                                                                  │
│  │   ├─ Check current status                                          │
│  │   └─ If both participants gone, set status = 'completed'          │
│  │                                                                      │
│  ├─ UPDATE sessions SET                                                │
│  │   ended_at = now(),                                                │
│  │   status = 'completed'                                             │
│  │   WHERE id = session_id                                            │
│  │                                                                      │
│  └─ INSERT INTO session_logs (                                         │
│       session_id, trainer_id, learner_id,                             │
│       connected_at, disconnected_at, duration,                        │
│       connection_quality, notes                                       │
│     )                                                                   │
│                                                                          │
│     ▼                                                                    │
│  SESSION REPORT GENERATION (Optional)                                  │
│  ├─ Check if session has required notes/feedback                     │
│  ├─ Calculate session metrics:                                        │
│  │   ├─ Duration = ended_at - started_at                             │
│  │   ├─ Participant count                                             │
│  │   ├─ Video quality score                                           │
│  │   └─ Issues encountered                                            │
│  │                                                                      │
│  └─ Create SessionReport record (if enabled)                          │
│     ├─ Trainer can view report later                                  │
│     └─ Head can view all session reports                              │
│                                                                          │
│     ▼                                                                    │
│  NOTIFICATION EMAILS (Optional)                                        │
│  ├─ Send email to trainer:                                            │
│  │   "Session ended: [learner_name], Duration: [time]"               │
│  │                                                                      │
│  └─ Send email to learner:                                            │
│      "Your session with [trainer_name] is complete"                   │
│                                                                          │
│     ▼                                                                    │
│  FRONTEND STATE UPDATE                                                  │
│  ├─ React component receives disconnect notification                   │
│  ├─ Update UI state: isConnected = false                              │
│  ├─ Show "Session Ended" message                                      │
│  ├─ Hide video elements                                               │
│  ├─ Offer options:                                                    │
│  │   ├─ View session notes                                            │
│  │   ├─ Provide feedback                                              │
│  │   └─ Return to dashboard                                           │
│  │                                                                      │
│  └─ Redirect to post-session page after 5 seconds                    │
│                                                                          │
└────────────────────────────────────────────────────────────────────────┘
```

---

## 5. WebRTC Media Streaming & Peer Connection Flow

```
┌────────────────────────────────────────────────────────────────────────┐
│             WEBRTC PEER CONNECTION & MEDIA STREAMING FLOW               │
├────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  ┌─ TRAINER SIDE ─────────────┐    ┌─ LEARNER SIDE ──────────────┐    │
│  │                             │    │                              │    │
│  │ 1. createPeerConnection()   │    │ 1. createPeerConnection()    │    │
│  │    ├─ RTCPeerConnection({   │    │    ├─ RTCPeerConnection({   │    │
│  │    │      iceServers: [     │    │    │      iceServers: [      │    │
│  │    │        {              │    │    │        {                 │    │
│  │    │          urls: 'stun:' │    │    │          urls: 'stun:'   │    │
│  │    │        },             │    │    │        },                │    │
│  │    │        {              │    │    │        {                 │    │
│  │    │          urls: 'turn:' │    │    │          urls: 'turn:'   │    │
│  │    │          username: ..  │    │    │          username: ..    │    │
│  │    │        }               │    │    │        }                 │    │
│  │    │      ]                 │    │    │      ]                  │    │
│  │    │    })                  │    │    │    })                   │    │
│  │    │                         │    │    │                         │    │
│  │    ├─ onicecandidate = () { │    │    ├─ onicecandidate = () { │    │
│  │    │   send ICE candidate   │    │    │   send ICE candidate    │    │
│  │    │ }                       │    │    │ }                       │    │
│  │    │                         │    │    │                         │    │
│  │    ├─ ontrack = (event) {   │    │    ├─ ontrack = (event) {    │    │
│  │    │   remoteStream.addTrack│    │    │   remoteStream.addTrack │    │
│  │    │   callback()            │    │    │   callback()             │    │
│  │    │ }                       │    │    │ }                       │    │
│  │    │                         │    │    │                         │    │
│  │    └─ onconnectionstatechange
│  │    │   (update UI state)     │    │    │   (update UI state)     │    │
│  │                             │    │                              │    │
│  │ 2. addLocalTracksToConnection
│  │    ├─ getSenders() ←─ ┐     │    │ 2. addLocalTracksToConnection
│  │    │                   │     │    │    ├─ getSenders() ←─ ┐    │    │
│  │    │  localStream      │     │    │    │                   │    │    │
│  │    │    ├─ video track │     │    │    │  localStream     │    │    │
│  │    │    └─ audio track │     │    │    │    ├─ video track│    │    │
│  │    │                   │     │    │    │    └─ audio track│    │    │
│  │    └─ addTrack(track)  │     │    │    │                   │    │    │
│  │       ├─ video sender  │     │    │    └─ addTrack(track)  │    │    │
│  │       └─ audio sender  │     │    │       ├─ video sender  │    │    │
│  │                         │    │    │       └─ audio sender  │    │    │
│  │ 3. CREATE OFFER         │    │    │                        │    │    │
│  │    └─ peerConnection    │    │    │ 3. RECEIVE OFFER       │    │    │
│  │       .createOffer({    │    │    │    └─ setRemoteDesc    │    │    │
│  │         offerToReceive  │    │    │       (trainer's SDP)   │    │    │
│  │         Audio: true,    │    │    │                        │    │    │
│  │         Video: true     │    │    │ 4. CREATE ANSWER       │    │    │
│  │       })                │    │    │    └─ peerConnection   │    │    │
│  │                         │    │    │       .createAnswer()  │    │    │
│  │    └─ setLocalDesc      │    │    │                        │    │    │
│  │       (offer SDP)       │    │    │    └─ setLocalDesc     │    │    │
│  │                         │    │    │       (answer SDP)     │    │    │
│  │ 4. SEND OFFER          │    │    │                        │    │    │
│  │    └─ ws.send({        │    │    │ 5. SEND ANSWER         │    │    │
│  │         type: 'offer'  │    │    │    └─ ws.send({        │    │    │
│  │         offer: {...}   │    │    │         type: 'answer' │    │    │
│  │       })               │    │    │         answer: {...}  │    │    │
│  │         │              │    │    │       })               │    │    │
│  │         └─────────────────────┼─────┬──────────────────► │    │    │
│  │                         │    │     │                     │    │    │
│  │    ◄─────────────────────────┼─────┘                      │    │    │
│  │ 5. RECEIVE ANSWER            │                           │    │    │
│  │    └─ setRemoteDesc          │                           │    │    │
│  │       (learner's SDP)        │                           │    │    │
│  │                             │                            │    │    │
│  │ 6. ICE CANDIDATE EXCHANGE   │                            │    │    │
│  │    onicecandidate → send    │                            │    │    │
│  │    receive → addIceCandidate │                            │    │    │
│  │         │                   │                            │    │    │
│  │         └───────────────────────────────────────────────┼────┘    │
│  │                             │    │                       │         │    │
│  │ 7. CONNECTION ESTABLISHED   │    │ 7. CONNECTION EST.    │         │    │
│  │    └─ iceConnectionState =  │    │    └─ iceConnectionState         │    │
│  │       'connected'            │    │       'connected'                │    │
│  │                             │    │                              │    │
│  └─────────────────────────────┘    └──────────────────────────────┘    │
│                                                                          │
│  ┌─────────────────────────────────────────────────────────────────┐    │
│  │        P2P CONNECTION ESTABLISHED - MEDIA FLOWING               │    │
│  │                                                                 │    │
│  │  TRAINER:                                                      │    │
│  │  ├─ Local video (self) → video element                         │    │
│  │  ├─ Remote video (learner) → video element                     │    │
│  │  ├─ Audio flowing bidirectional                                │    │
│  │  └─ Metadata: ontrack fires, adds tracks to remote stream      │    │
│  │                                                                 │    │
│  │  LEARNER:                                                      │    │
│  │  ├─ Local video (self) → NOT displayed (minimize distraction)  │    │
│  │  ├─ Remote video (trainer) → video element                     │    │
│  │  ├─ Audio flowing bidirectional                                │    │
│  │  └─ Metadata: ontrack fires, adds tracks to remote stream      │    │
│  │                                                                 │    │
│  │  NETWORK PATHS:                                                │    │
│  │  ├─ Type 1 (Best): Direct P2P (host-to-host)                  │    │
│  │  ├─ Type 2 (Good): STUN NAT traversal (same network)           │    │
│  │  ├─ Type 3 (OK): TURN relay (firewall bypass)                 │    │
│  │  └─ TURN Credentials: hurshbin / SecureTurnPass2024!           │    │
│  │                                                                 │    │
│  │  HEARTBEAT (ongoing every 30s):                                │    │
│  │  ├─ Backend sends ping                                         │    │
│  │  ├─ Client responds with pong                                  │    │
│  │  └─ If no pong for 90s → Close connection                     │    │
│  └─────────────────────────────────────────────────────────────────┘    │
│                                                                          │
└────────────────────────────────────────────────────────────────────────┘
```

---

## 4. Session Join & WebSocket Connection Flow

```
┌────────────────────────────────────────────────────────────────────────┐
│              SESSION JOIN & WEBSOCKET CONNECTION FLOW                   │
├────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  TRAINER SIDE                              │      LEARNER SIDE          │
│                                            │                            │
│  1. Navigate to /trainer-session/{id}      │      1. Navigate to        │
│  2. TrainerSession.jsx mounts             │         /learner-session/{id}
│                                            │      2. LearnerSession.jsx  │
│                                            │         mounts              │
│     ▼                                      │          ▼                 │
│  initializeWebRTC()                       │      initializeWebRTC()    │
│     │                                      │          │                 │
│     ├─ requestMediaAccess()               │      ├─ requestMediaAccess()
│     │   └─ getUserMedia({                 │      │   └─ getUserMedia({  │
│     │        video: {720p@30fps},         │      │        video: {720p},│
│     │        audio: {...}                 │      │        audio: {...}  │
│     │      })                             │      │      })              │
│     │                                      │      │                     │
│     ├─ Display local video in preview     │      ├─ (Not displayed)    │
│     │                                      │      │                     │
│     └─ createVideoConferenceService()    │      └─ createVideoService()
│        with role='trainer'                │         with role='learner'│
│                                            │                            │
│     ▼                                      │          ▼                 │
│  connectWebSocket()                       │      connectWebSocket()    │
│     │                                      │          │                 │
│     ├─ URL: wss://backend/ws/rtc/{id}    │      ├─ URL: wss://backend │
│     │                                      │      │       /ws/rtc/{id}  │
│     ├─ onopen: resolve()                  │      ├─ onopen: resolve()  │
│     │                                      │      │                     │
│     └─ onmessage: handleSignaling()       │      └─ onmessage:         │
│                                            │         handleSignaling()  │
│     ▼                                      │          ▼                 │
│  SEND JOIN MESSAGE                        │      SEND JOIN MESSAGE     │
│  {                                        │      {                     │
│    type: "join",                         │        type: "join",       │
│    role: "trainer",                      │        role: "learner",    │
│    sessionId: "{session_id}"             │        sessionId: "..."    │
│  }                                        │      }                     │
│     │                                      │          │                 │
│     └─────────────────────┬────────────────┼──────────┘                 │
│                           │                │                            │
│                           ▼                ▼                            │
│                    BACKEND (/ws/rtc/{id})                              │
│                    ├─ Accept WebSocket connection                      │
│                    ├─ Receive join message                             │
│                    ├─ Create ConnectionInfo(role, session_id)          │
│                    ├─ Store: rtc_connections[{id}][{role}]            │
│                    ├─ Start heartbeat task (ping every 30s)           │
│                    ├─ Send "connected" confirmation                    │
│                    │                                                    │
│                    ├─ Check if both participants connected             │
│                    │   if len(rtc_connections[{id}]) == 2:             │
│                    │       send "peer_ready" to trainer                │
│                    │                                                    │
│                    └─ Message relay loop:                              │
│                       while conn_info.is_alive:                        │
│                           msg = await ws.receive_json()                │
│                           if msg.type == "ping":                       │
│                               send_pong()                              │
│                           else:                                        │
│                               relay_to_peer(msg)                       │
│                                                                          │
│     ▼                                      │          ▼                 │
│  Receive "connected"                      │      Receive "connected"  │
│  Receive "peer_ready"                     │                            │
│  └─ peerConnection.readyState = true     │                            │
│                                            │                            │
│     ▼                                      │          ▼                 │
│  createOffer()                             │      await handleOffer()  │
│  ├─ offer = peerConnection                │      ├─ setRemoteDesc     │
│  │          .createOffer()                 │      │   (trainer's offer)│
│  ├─ peerConnection.setLocalDesc(offer)   │      ├─ createAnswer()    │
│  │                                        │      ├─ setLocalDesc      │
│  └─ Send offer to learner via WS         │      │   (answer)          │
│                                            │      └─ Send answer to    │
│                                            │         trainer via WS    │
│                                            │                            │
│     ◄─── Receive "answer" ─────────────────┼─────────────────            │
│     │                                      │                            │
│     ├─ setRemoteDescription(answer)       │                            │
│     │                                      │                            │
│     ▼                                      │          ▼                 │
│  ICE Candidate Exchange                   │      ICE Candidate Exch.  │
│  ├─ onicecandidate event fired            │      ├─ onicecandidate    │
│  ├─ Send each candidate to learner        │      │   event fired       │
│  │                                        │      ├─ Send each to      │
│  └─ Add received candidates from learner │      │   trainer          │
│                                            │      └─ Add received      │
│                                            │         from trainer      │
│                                                                          │
└────────────────────────────────────────────────────────────────────────┘

**Timeline:**
T+0s:  Trainer clicks session link
T+1s:  Trainer WebSocket connects
T+2s:  Learner clicks session link  
T+3s:  Learner WebSocket connects
T+4s:  Backend sends "peer_ready" to trainer
T+5s:  Trainer creates offer
T+6s:  Learner receives offer, creates answer
T+7s:  Trainer receives answer
T+8s:  ICE candidates exchanged (multiple messages)
T+12s: P2P connection established
T+13s: Video/audio flowing directly between peers
```

---

## 3. Session Creation Flow (Head Creates Training Session)

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    SESSION CREATION FLOW                                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                           │
│  HEAD DASHBOARD                                                           │
│  └─ Click "Create New Session"                                           │
│     ├─ Opens modal/form                                                  │
│     └─ Fields:                                                           │
│        ├─ trainer_id (Select from trainers list)                        │
│        ├─ learner_id (Select from learners list)                        │
│        ├─ scheduled_datetime (Date & time picker)                       │
│        ├─ session_type (Training/Coach/Guest)                           │
│        ├─ topic (Session topic/description)                             │
│        └─ notes (Optional additional notes)                             │
│                      │                                                    │
│                      ▼                                                    │
│  FRONTEND VALIDATION                                                      │
│  ├─ Check trainer selected                                              │
│  ├─ Check learner selected                                              │
│  ├─ Check future date/time                                              │
│  └─ Show validation errors if any                                       │
│                      │                                                    │
│                      ▼                                                    │
│  POST /sessions/create                                                   │
│  └─ {                                                                   │
│      trainer_id: 5,                                                     │
│      learner_id: 23,                                                    │
│      scheduled_datetime: "2024-02-15T14:00:00",                         │
│      topic: "DLB Assessment",                                           │
│      notes: "Follow-up from last session"                               │
│     }                                                                    │
│                      │                                                    │
│                      ▼                                                    │
│  BACKEND SESSION CREATION                                                │
│  ├─ Verify JWT token & head authorization                              │
│  ├─ Validate trainer exists & is active                                │
│  ├─ Validate learner exists & is active                                │
│  ├─ Generate unique session_id (UUID)                                  │
│  ├─ Check trainer availability at scheduled_datetime                   │
│  └─ Check learner availability at scheduled_datetime                   │
│                      │                                                    │
│                      ▼                                                    │
│  DATABASE RECORD CREATION                                                │
│  ├─ INSERT INTO sessions (                                              │
│  │      id, trainer_id, learner_id, head_id,                           │
│  │      scheduled_datetime, created_datetime,                          │
│  │      status='scheduled', session_type='training',                   │
│  │      topic, notes                                                   │
│  │  ) RETURNING session_id                                             │
│  │                                                                      │
│  └─ INSERT INTO session_participants (                                  │
│         session_id, user_id, role, joined_at=NULL                      │
│      ) ← For each trainer and learner                                   │
│                      │                                                    │
│                      ▼                                                    │
│  NOTIFICATION SYSTEM                                                     │
│  ├─ Send email to trainer:                                             │
│  │   "Session scheduled for [datetime]"                                │
│  │                                                                      │
│  └─ Send email to learner:                                             │
│      "Your session is scheduled for [datetime]"                        │
│                      │                                                    │
│                      ▼                                                    │
│  RESPONSE TO HEAD                                                        │
│  ├─ Return { session_id, status, trainer_name, learner_name }         │
│  ├─ Update sessions list in dashboard                                  │
│  ├─ Show success toast                                                 │
│  └─ Offer option to immediately join if time has passed               │
│                                                                           │
└─────────────────────────────────────────────────────────────────────────┘
```

**Session States:**
- `scheduled` - Session created, waiting to start
- `in-progress` - At least one participant has joined
- `completed` - Both participants disconnected
- `cancelled` - Session was cancelled
- `no-show` - Session time passed, no one joined

---

## 2. Authentication & Login Flow

```
┌──────────────────────────────────────────────────────────────────────┐
│                     AUTHENTICATION FLOW                               │
├──────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  LOGIN PAGE                                                            │
│  └─ User selects role from options                                    │
│     ├─ Admin                                                          │
│     ├─ Head                                                           │
│     ├─ Trainer                                                        │
│     ├─ Learner                                                        │
│     └─ Coach                                                          │
│                      │                                                 │
│  └─ User enters passcode (6 digits)                                  │
│                      │                                                 │
│                      ▼                                                 │
│  FRONTEND VALIDATION                                                   │
│  ├─ Check role selected                                              │
│  ├─ Check passcode length (6)                                        │
│  └─ Check passcode numeric only                                      │
│                      │                                                 │
│                      ▼                                                 │
│  POST /auth/user/login {role, passcode}                              │
│                      │                                                 │
│                      ▼                                                 │
│  BACKEND AUTHENTICATION (Rate Limited)                                │
│  ├─ Rate limit check (max 5 attempts/minute)                         │
│  │   └─ If exceeded: return 429 Too Many Requests                    │
│  │                                                                    │
│  └─ Query users table                                                │
│     └─ SELECT * FROM users WHERE role='{role}'                      │
│        AND passcode_hash=hash(passcode)                              │
│                      │                                                 │
│        ┌─────────────┴──────────────┐                                 │
│        │                            │                                 │
│   ✓ USER FOUND              ✗ USER NOT FOUND                        │
│        │                            │                                 │
│        ▼                            ▼                                 │
│  Check is_active flag         Return 401 Unauthorized                │
│        │                            │                                 │
│   ┌────┴────┐                       │                                 │
│   │         │                       │                                 │
│ ✓ True   ✗ False                    │                                 │
│   │         │                       │                                 │
│   │         └──→ Return 401         │                                 │
│   │              Inactive Account   │                                 │
│   │                                 │                                 │
│   ▼                                 ▼                                 │
│  Create JWT Token              Frontend Error Toast                   │
│  └─ Payload: {                  └─ Display error message               │
│       sub: user_id,                                                   │
│       role: role,                                                     │
│       exp: now + 24h                                                  │
│     }                                                                  │
│                      │                                                 │
│                      ▼                                                 │
│  Return Token Response                                                │
│  └─ {                                                                │
│      access_token: "eyJhbGc...",                                     │
│      token_type: "bearer",                                           │
│      role: "head",                                                   │
│      user_id: 42,                                                    │
│      name: "John Doe"                                                │
│     }                                                                │
│                      │                                                │
│                      ▼                                                │
│  FRONTEND STORAGE                                                      │
│  ├─ localStorage.setItem('access_token', token)                      │
│  ├─ localStorage.setItem('user_data', JSON.stringify(user))          │
│  └─ AuthContext.setUser(user)                                        │
│                      │                                                │
│                      ▼                                                │
│  REDIRECT TO DASHBOARD                                               │
│  ├─ /admin/dashboard (Admin)                                         │
│  ├─ /head/dashboard (Head)                                           │
│  ├─ /trainer/dashboard (Trainer)                                     │
│  ├─ /learner/dashboard (Learner)                                     │
│  └─ /coach/dashboard (Coach)                                         │
│                                                                        │
└──────────────────────────────────────────────────────────────────────┘
```

**Authentication Flow Details:**
- **Rate Limiting:** 5 attempts per minute per IP
- **Token Expiry:** 24 hours
- **Passcode Storage:** Hashed + Encrypted (double protection)
- **JWT Verification:** Token validated on every API call

---

## 1. User Creation Flow (Admin → Head/Trainer/Learner)

```
┌─────────────────────────────────────────────────────────────────────┐
│                      USER CREATION FLOW                              │
├─────────────────────────────────────────────────────────────────────┤
│                                                                       │
│  ADMIN PORTAL                                                         │
│  └─ Navigate to "Register Head"/"Register Trainer"                   │
│     ├─ Fill Form: name, email, age, address, contact_no             │
│     └─ Submit Registration Request                                   │
│                      │                                                │
│                      ▼                                                │
│  FRONTEND VALIDATION                                                  │
│  └─ Validate all fields                                              │
│     ├─ Email format check                                            │
│     ├─ Contact number validation                                     │
│     ├─ Age range (18-100)                                            │
│     └─ Required field checks                                         │
│                      │                                                │
│                      ▼                                                │
│  BACKEND PROCESSING (adminService.registerHead)                      │
│  └─ POST /admin/heads                                                │
│     ├─ Generate unique 6-digit passcode                              │
│     ├─ Encrypt passcode for storage                                  │
│     ├─ Hash passcode for verification                                │
│     └─ Create User & HeadProfile records                             │
│                      │                                                │
│                      ▼                                                │
│  DATABASE OPERATIONS                                                  │
│  ├─ INSERT INTO users (                                              │
│  │      name, email, passcode_hash, passcode_encrypted,              │
│  │      role='HEAD', is_active=true                                  │
│  │  )                                                                 │
│  │                                                                    │
│  └─ INSERT INTO head_profile (                                       │
│         user_id, age, address, contact_no, qualification,            │
│         experience, date_of_joining, ...                             │
│      )                                                                │
│                      │                                                │
│                      ▼                                                │
│  RESPONSE TO ADMIN                                                    │
│  └─ Return { head_id, user_id, passcode, email }                    │
│     ├─ Display success message with passcode                         │
│     ├─ Display copy-to-clipboard option                              │
│     └─ Redirect to manage-heads page                                 │
│                                                                       │
└─────────────────────────────────────────────────────────────────────┘
```

**Key Database Tables:**
- `users` - Centralized user table (id, name, email, role, passcode_hash)
- `head_profile` - Head-specific profile (user_id, age, address, contact_no, ...)
- `trainer_profile` - Trainer-specific profile (user_id, specialization, ...)
- `learner_profile` - Learner-specific profile (user_id, dob, age, ...)

---

In [ ]:
# Install and import required libraries
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

# For creating structured diagrams
print("✓ Libraries imported successfully")
print("Ready to create application flow diagrams")

# HurshBinTech PWA - Complete Application Flow Diagram

## Overview
This notebook provides a comprehensive low-level flow diagram of the complete HurshBinTech application, starting from user creation through session management and WebRTC media streaming.

### Flow Phases:
1. **User Creation & Registration** - Admin creates user (Head/Trainer/Learner)
2. **Authentication & Login** - User authenticates with passcode
3. **Session Initialization** - Head creates training session with participants
4. **WebSocket Connection** - Participants connect via WebSocket signaling
5. **WebRTC Peer Setup** - Offer/answer negotiation and ICE candidate exchange
6. **Media Streaming** - Audio/video transmission through WebRTC P2P
7. **Session Termination** - Graceful disconnection and resource cleanup

---